# init

> Router initialization for the structure decomposition workflow

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
from typing import List, Dict, Any, Tuple, Callable

from fasthtml.common import APIRouter

from cjm_fasthtml_card_stack.core.models import CardStackUrls

from cjm_fasthtml_workflow_transcript_decomp.routes.models import (
    DecompUrls, SelectionUrls, AlignmentUrls,
)

# Import core router assembly
from cjm_fasthtml_workflow_transcript_decomp.routes.core.init import init_core_routers

# Import Phase 1 (selection) handlers from sub-package
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.queue import (
    _handle_selection_add,
    _handle_selection_remove,
    _handle_selection_reorder,
    _handle_selection_clear,
    _handle_selection_select_all,
    _handle_selection_preview,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.filtering import (
    _handle_source_filter,
    _handle_grouping_change,
    _handle_selection_toggle_focused,
    _handle_keyboard_reorder,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.local_files import (
    _handle_browse_directory,
    _handle_add_external_source,
    _handle_remove_external_source,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.tabs import (
    _handle_tab_switch,
)

# Import Phase 2 (decomposition) handlers — card stack UI operations
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.card_stack import (
    _handle_decomp_navigate,
    _handle_decomp_enter_split_mode,
    _handle_decomp_exit_split_mode,
    _handle_decomp_update_viewport,
    _handle_decomp_save_width,
)

# Import Phase 2 (decomposition) handlers — workflow-specific operations
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.handlers import (
    _handle_decomp_init,
    _handle_decomp_split,
    _handle_decomp_merge,
    _handle_decomp_undo,
    _handle_decomp_reset,
    _handle_decomp_ai_split,
)

# Import Phase 2 (alignment) handlers — card stack UI operations
from cjm_fasthtml_workflow_transcript_decomp.routes.alignment.card_stack import (
    _handle_align_navigate,
    _handle_align_update_viewport,
    _handle_align_save_width,
)

# Import Phase 2 (alignment) handlers — workflow-specific operations
from cjm_fasthtml_workflow_transcript_decomp.routes.alignment.handlers import (
    _handle_align_init,
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Selection Router

Phase 1 routes for source selection and queue management.

In [ ]:
#| export
def init_selection_router(
    workflow: "StructureDecompWorkflow",  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/selection")
) -> Tuple[APIRouter, SelectionUrls, Dict[str, Callable]]:  # (router, urls, route_dict)
    """Initialize Phase 1 selection routes."""
    router = APIRouter(prefix=prefix)

    # -------------------------------------------------------------------------
    # Queue Management
    # -------------------------------------------------------------------------

    @router
    def add(request, sess, record_id: str, provider_id: str):
        """Add a source to the selection queue."""
        return _handle_selection_add(
            workflow, request, sess, record_id, provider_id, urls=urls,
        )

    @router
    def remove(request, sess, record_id: str):
        """Remove a source from the selection queue."""
        return _handle_selection_remove(
            workflow, request, sess, record_id, urls=urls,
        )

    @router
    async def reorder(request, sess):
        """Reorder items in the selection queue."""
        return await _handle_selection_reorder(
            workflow, request, sess, urls=urls,
        )

    @router
    def clear(request, sess):
        """Clear all items from the selection queue."""
        return _handle_selection_clear(
            workflow, request, sess, urls=urls,
        )

    @router
    def select_all(request, sess, group_key: str, grouping_mode: str = "media_path"):
        """Select all transcriptions for a given group."""
        return _handle_selection_select_all(
            workflow, request, sess, group_key, grouping_mode, urls=urls,
        )

    @router
    def preview(request, record_id: str, provider_id: str):
        """Get preview content for a selected source."""
        return _handle_selection_preview(workflow, request, record_id, provider_id)

    # -------------------------------------------------------------------------
    # Keyboard and Filtering
    # -------------------------------------------------------------------------

    @router
    def toggle_focused(request, sess, record_id: str, provider_id: str):
        """Toggle selection of the focused row (keyboard shortcut)."""
        return _handle_selection_toggle_focused(
            workflow, request, sess, record_id, provider_id, urls=urls,
        )

    @router
    def keyboard_reorder(request, sess, record_id: str, provider_id: str, direction: str):
        """Move an item up or down in the queue via keyboard (Shift+Up/Down)."""
        return _handle_keyboard_reorder(
            workflow, request, sess, record_id, provider_id, direction, urls=urls,
        )

    @router
    def filter(request, sess, search: str = ""):
        """Filter source list by search term."""
        return _handle_source_filter(
            workflow, request, sess, search, urls=urls,
        )

    @router
    def grouping_change(request, sess, grouping_mode: str):
        """Change the grouping mode for the source list."""
        return _handle_grouping_change(
            workflow, request, sess, grouping_mode, urls=urls,
        )

    # -------------------------------------------------------------------------
    # Local Files Browser
    # -------------------------------------------------------------------------

    @router
    def browse_directory(request, sess, path: str):
        """Browse a directory in the local files browser."""
        return _handle_browse_directory(
            workflow, request, sess, path, urls=urls,
        )

    @router
    def add_external(request, sess, path: str):
        """Add an external database source (select_url target from file browser)."""
        return _handle_add_external_source(
            workflow, request, sess, path, urls=urls,
        )

    @router
    def remove_external(request, sess, db_path: str):
        """Remove an external database source."""
        return _handle_remove_external_source(
            workflow, request, sess, db_path, urls=urls,
        )

    # -------------------------------------------------------------------------
    # Tab Switching
    # -------------------------------------------------------------------------

    @router
    def tab_switch(request, sess, direction: str):
        """Switch between source tabs."""
        return _handle_tab_switch(
            workflow, request, sess, direction, urls=urls,
        )

    # -------------------------------------------------------------------------
    # URL Bundle and Route Dict
    # -------------------------------------------------------------------------

    urls = SelectionUrls(
        add=add.to(),
        remove=remove.to(),
        reorder=reorder.to(),
        clear=clear.to(),
        select_all=select_all.to(),
        preview=preview.to(),
        toggle_focused=toggle_focused.to(),
        keyboard_reorder=keyboard_reorder.to(),
        filter=filter.to(),
        grouping_change=grouping_change.to(),
        browse_directory=browse_directory.to(),
        add_external=add_external.to(),
        remove_external=remove_external.to(),
        tab_switch=tab_switch.to(),
    )

    routes = {
        "add": add,
        "remove": remove,
        "reorder": reorder,
        "clear": clear,
        "select_all": select_all,
        "preview": preview,
        "toggle_focused": toggle_focused,
        "keyboard_reorder": keyboard_reorder,
        "filter": filter,
        "grouping_change": grouping_change,
        "browse_directory": browse_directory,
        "add_external": add_external,
        "remove_external": remove_external,
        "tab_switch": tab_switch,
    }

    return router, urls, routes

## Decomposition Router

Phase 2 left column routes for text segmentation operations.

In [ ]:
#| export
def init_decomposition_router(
    workflow: "StructureDecompWorkflow",  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/decomp")
) -> Tuple[APIRouter, DecompUrls, Dict[str, Callable]]:  # (router, urls, route_dict)
    """Initialize Phase 2 decomposition routes."""
    router = APIRouter(prefix=prefix)

    # -------------------------------------------------------------------------
    # Workflow Operations
    # -------------------------------------------------------------------------

    @router
    async def init(request, sess):
        """Initialize segments from Phase 1 selected sources."""
        return await _handle_decomp_init(workflow, request, sess, urls=urls)

    @router
    async def split(request, sess, segment_index: int):
        """Split a segment at the specified word position."""
        return await _handle_decomp_split(
            workflow, request, sess, segment_index, urls=urls,
        )

    @router
    def merge(request, sess, segment_index: int):
        """Merge a segment with the previous segment."""
        return _handle_decomp_merge(
            workflow, request, sess, segment_index, urls=urls,
        )

    @router
    def undo(request, sess):
        """Undo the last decomposition operation."""
        return _handle_decomp_undo(workflow, request, sess, urls=urls)

    @router
    def reset(request, sess):
        """Reset segments to the initial NLTK split result."""
        return _handle_decomp_reset(workflow, request, sess, urls=urls)

    @router
    async def ai_split(request, sess):
        """Re-run AI (NLTK) sentence splitting on all current text."""
        return await _handle_decomp_ai_split(workflow, request, sess, urls=urls)

    # -------------------------------------------------------------------------
    # Split Mode
    # -------------------------------------------------------------------------

    @router
    def enter_split(request, sess, segment_index: int):
        """Enter split mode for a specific segment."""
        return _handle_decomp_enter_split_mode(
            workflow, request, sess, segment_index, urls=urls,
        )

    @router
    def exit_split(request, sess):
        """Exit split mode."""
        return _handle_decomp_exit_split_mode(workflow, request, sess, urls=urls)

    # -------------------------------------------------------------------------
    # Viewport and Width
    # -------------------------------------------------------------------------

    @router
    async def update_viewport(request, sess, visible_count: int):
        """Update viewport with new card count (full outerHTML swap)."""
        return await _handle_decomp_update_viewport(
            workflow, request, sess, visible_count, urls=urls,
        )

    @router
    def save_width(request, sess, card_width: int):
        """Save card stack width to server state."""
        return _handle_decomp_save_width(workflow, sess, card_width)

    # -------------------------------------------------------------------------
    # Navigation
    # -------------------------------------------------------------------------

    @router
    def nav_up(request, sess):
        """Navigate to previous segment."""
        return _handle_decomp_navigate(workflow, sess, direction="up", urls=urls)

    @router
    def nav_down(request, sess):
        """Navigate to next segment."""
        return _handle_decomp_navigate(workflow, sess, direction="down", urls=urls)

    @router
    def nav_first(request, sess):
        """Navigate to first segment."""
        return _handle_decomp_navigate(workflow, sess, direction="first", urls=urls)

    @router
    def nav_last(request, sess):
        """Navigate to last segment."""
        return _handle_decomp_navigate(workflow, sess, direction="last", urls=urls)

    @router
    def nav_page_up(request, sess):
        """Navigate up by page."""
        return _handle_decomp_navigate(workflow, sess, direction="page_up", urls=urls)

    @router
    def nav_page_down(request, sess):
        """Navigate down by page."""
        return _handle_decomp_navigate(workflow, sess, direction="page_down", urls=urls)

    # -------------------------------------------------------------------------
    # URL Bundle and Route Dict
    # -------------------------------------------------------------------------

    urls = DecompUrls(
        card_stack=CardStackUrls(
            nav_up=nav_up.to(),
            nav_down=nav_down.to(),
            nav_first=nav_first.to(),
            nav_last=nav_last.to(),
            nav_page_up=nav_page_up.to(),
            nav_page_down=nav_page_down.to(),
            update_viewport=update_viewport.to(),
            save_width=save_width.to(),
        ),
        split=split.to(),
        merge=merge.to(),
        enter_split=enter_split.to(),
        exit_split=exit_split.to(),
        reset=reset.to(),
        ai_split=ai_split.to(),
        undo=undo.to(),
        init=init.to(),
    )

    routes = {
        "init": init,
        "split": split,
        "merge": merge,
        "undo": undo,
        "reset": reset,
        "ai_split": ai_split,
        "enter_split": enter_split,
        "exit_split": exit_split,
        "update_viewport": update_viewport,
        "save_width": save_width,
        "nav_up": nav_up,
        "nav_down": nav_down,
        "nav_first": nav_first,
        "nav_last": nav_last,
        "nav_page_up": nav_page_up,
        "nav_page_down": nav_page_down,
    }

    return router, urls, routes

## Alignment Router

Phase 2 right column routes for VAD audio alignment.

In [ ]:
#| export
def init_alignment_router(
    workflow: "StructureDecompWorkflow",  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/align")
    audio_src_url: str,  # URL for audio_src route (from core router)
) -> Tuple[APIRouter, AlignmentUrls, Dict[str, Callable]]:  # (router, urls, route_dict)
    """Initialize Phase 2 alignment routes."""
    router = APIRouter(prefix=prefix)

    # -------------------------------------------------------------------------
    # Workflow Operations
    # -------------------------------------------------------------------------

    @router
    async def init(request, sess):
        """Initialize alignment from audio file via VAD plugin."""
        return await _handle_align_init(workflow, request, sess, urls=urls)

    # -------------------------------------------------------------------------
    # Navigation
    # -------------------------------------------------------------------------

    @router
    def nav_up(request, sess):
        """Navigate to previous VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="up", urls=urls)

    @router
    def nav_down(request, sess):
        """Navigate to next VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="down", urls=urls)

    @router
    def nav_first(request, sess):
        """Navigate to first VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="first", urls=urls)

    @router
    def nav_last(request, sess):
        """Navigate to last VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="last", urls=urls)

    @router
    def nav_page_up(request, sess):
        """Navigate up by page in alignment."""
        return _handle_align_navigate(workflow, sess, direction="page_up", urls=urls)

    @router
    def nav_page_down(request, sess):
        """Navigate down by page in alignment."""
        return _handle_align_navigate(workflow, sess, direction="page_down", urls=urls)

    # -------------------------------------------------------------------------
    # Viewport and Width
    # -------------------------------------------------------------------------

    @router
    async def update_viewport(request, sess, visible_count: int):
        """Update alignment viewport with new card count."""
        return await _handle_align_update_viewport(
            workflow, request, sess, visible_count, urls=urls,
        )

    @router
    def save_width(request, sess, card_width: int):
        """Save alignment card stack width to server state."""
        return _handle_align_save_width(workflow, sess, card_width)

    # -------------------------------------------------------------------------
    # URL Bundle and Route Dict
    # -------------------------------------------------------------------------

    urls = AlignmentUrls(
        card_stack=CardStackUrls(
            nav_up=nav_up.to(),
            nav_down=nav_down.to(),
            nav_first=nav_first.to(),
            nav_last=nav_last.to(),
            nav_page_up=nav_page_up.to(),
            nav_page_down=nav_page_down.to(),
            update_viewport=update_viewport.to(),
            save_width=save_width.to(),
        ),
        init=init.to(),
        audio_src=audio_src_url,
    )

    routes = {
        "init": init,
        "nav_up": nav_up,
        "nav_down": nav_down,
        "nav_first": nav_first,
        "nav_last": nav_last,
        "nav_page_up": nav_page_up,
        "nav_page_down": nav_page_down,
        "update_viewport": update_viewport,
        "save_width": save_width,
    }

    return router, urls, routes

## Router Assembly

The `init_routers` function calls all focused router initializers and wires up cross-router dependencies.

In [ ]:
#| export
def init_routers(
    workflow: "StructureDecompWorkflow",  # The workflow instance
) -> List[APIRouter]:  # List of configured routers
    """Initialize and return all workflow routers."""
    base_prefix = workflow.config.route_prefix

    # Initialize focused routers
    core_routers, core_routes = init_core_routers(
        workflow, f"{base_prefix}/core"
    )
    selection_router, selection_urls, selection_routes = init_selection_router(
        workflow, f"{base_prefix}/selection"
    )
    decomp_router, decomp_urls, decomp_routes = init_decomposition_router(
        workflow, f"{base_prefix}/decomp"
    )
    align_router, align_urls, align_routes = init_alignment_router(
        workflow, f"{base_prefix}/align",
        audio_src_url=core_routes["audio_src"].to(),
    )

    # Store URL bundles on workflow for renderer access
    workflow._selection_urls = selection_urls
    workflow._decomp_urls = decomp_urls
    workflow._align_urls = align_urls
    workflow._switch_chrome_url = core_routes["switch_chrome"].to()

    # Store route dicts on workflow
    workflow._core_routes = core_routes
    workflow._selection_routes = selection_routes
    workflow._decomposition_routes = decomp_routes
    workflow._alignment_routes = align_routes

    return core_routers + [selection_router, decomp_router, align_router]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()